# BollBot – train, evaluate and export (Colab)

Runtime → Change runtime type → **GPU**. Upload `bollbot.zip` and your `raw/` dataset (one folder per farm) to Google Drive under `MyDrive/bollbot/` first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!unzip -oq /content/drive/MyDrive/bollbot/bollbot.zip -d /content
!pip -q install ultralytics pyyaml

## 1. Validate the raw dataset
Fix every ERROR before continuing. WARNs are worth a look.

In [ ]:
!python -m bollbot.ml.dataset /content/drive/MyDrive/bollbot/raw --json /content/drive/MyDrive/bollbot/dataset_report.json

## 2. Split by farm
Edit the farm IDs. The test farm must be one the model never sees in training. Add `--pretrain <public_dir>` to use public images in training only. `--overwrite` replaces the previous split in `/content/ds`.

In [ ]:
!python -m bollbot.ml.split /content/drive/MyDrive/bollbot/raw /content/ds --train F01 F02 --val F03 --test F04 --overwrite

## 3. Train (E1)
Settings come from `bollbot/config/ml.yaml` (YOLOv8n, 416 px, up to 150 epochs, early stopping).

In [ ]:
!python -m bollbot.ml.train /content/ds/data.yaml --name v1 --project /content/drive/MyDrive/bollbot/runs

## 4. Evaluate on the held-out test farm
Writes `eval_test.json` – the only source of numbers for the report and dashboard.

In [ ]:
!python -m bollbot.ml.evaluate /content/drive/MyDrive/bollbot/runs/v1/weights/best.pt /content/ds/data.yaml

## 5. Export
NCNN (Pi fallback), TFLite int8 (phone), OpenVINO. For the OAK-D Lite, convert `best.pt` at tools.luxonis.com (YOLOv8, 416).

In [ ]:
!python -m bollbot.ml.export /content/drive/MyDrive/bollbot/runs/v1/weights/best.pt /content/ds/data.yaml

## 6. Optional ablation (E1)
Same data, different input size. Compare `eval_test.json` files.

In [ ]:
!python -m bollbot.ml.train /content/ds/data.yaml --name v1-512 --imgsz 512 --project /content/drive/MyDrive/bollbot/runs